<a href="https://colab.research.google.com/github/nayakamhrudaya/GenAI/blob/main/Q%26AChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# INSTALL
# =========================
!pip install chromadb pypdf openai gradio

# =========================
# IMPORTS
# =========================
import os
from google.colab import files
from pypdf import PdfReader
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
import gradio as gr

# =========================
# API KEY
# =========================
os.environ["OPENAI_API_KEY"] = "GENERATED_API_KEY"
# =========================
# UPLOAD PDF
# =========================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# =========================
# READ PDF
# =========================
reader = PdfReader(file_name)
text = ""

for page in reader.pages:
    content = page.extract_text()
    if content:
        text += content

print("PDF loaded ✅")

# =========================
# CHUNKING
# =========================
def chunk_text(text, chunk_size=800, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

chunks = chunk_text(text)

# =========================
# VECTOR DB
# =========================
client = chromadb.Client()

embedding_function = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name="text-embedding-3-small"
)

collection = client.create_collection(
    name="rag_pdf",
    embedding_function=embedding_function
)

collection.add(
    documents=chunks,
    ids=[str(i) for i in range(len(chunks))]
)

# =========================
# LLM
# =========================
client_llm = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# =========================
# MEMORY
# =========================
chat_history = []

# =========================
# RETRIEVE
# =========================
def retrieve(query):
    results = collection.query(
        query_texts=[query],
        n_results=4
    )
    return results["documents"][0]

# =========================
# GENERATE ANSWER
# =========================
def generate_answer(query, context_chunks):
    context = "\n".join(context_chunks)

    memory_context = "\n".join(
        [f"Q: {q} A: {a}" for q, a in chat_history]
    )

    prompt = f"""
You are a helpful assistant.

Use the context and past conversation.

Conversation so far:
{memory_context}

Context:
{context}

Question:
{query}
"""

    response = client_llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content

# =========================
# SUMMARY OF CHAT
# =========================
def summarize_chat():
    full_chat = "\n".join(
        [f"Q: {q}\nA: {a}" for q, a in chat_history]
    )

    prompt = f"""
Summarize this conversation clearly:

{full_chat}
"""

    res = client_llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return res.choices[0].message.content

# =========================
# SHOW HISTORY
# =========================
def show_history():
    return "\n".join([f"{i+1}. {q}" for i, (q, _) in enumerate(chat_history)])

# =========================
# MAIN CHAT FUNCTION
# =========================
def chatbot_fn(message, history):
    global chat_history

    msg = message.lower().strip()

    if msg == "summary":
        return summarize_chat()

    elif msg == "history":
        return show_history()

    else:
        context_chunks = retrieve(message)
        answer = generate_answer(message, context_chunks)

        chat_history.append((message, answer))
        return answer

# =========================
# GRADIO UI
# =========================
gr.ChatInterface(
    fn=chatbot_fn,
    title="📄 Smart PDF Memory Chatbot",
    description="""
- Ask questions from PDF
- Type 'summary' → summarize all answers
- Type 'history' → see all questions asked
"""
).launch()

Saving AIOverview.pdf to AIOverview (1).pdf
PDF loaded ✅


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f60d1ad80b717cc88f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
